# Results

Thin notebook: it only reads `results/metrics/<run_id>/` (written by `scv analyze`) and draws the figures with `scv.analysis`. No metric is computed here.

Set `RUN_DIRS` to one or more metrics directories; several runs (elections) are shown as panels.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from scv import analysis

# One directory per run, e.g. ["../results/metrics/anes2024_pilot_v1"].
RUN_DIRS = [p for p in os.environ.get("SCV_RUN_DIRS", "").split(os.pathsep) if p] or sorted(
    str(p) for p in Path("../results/metrics").glob("*") if (p / "summary.csv").exists()
)
tables = [analysis.read_outputs(d) for d in RUN_DIRS]
metrics = pd.concat([t["metrics"] for t in tables], ignore_index=True)
ncurves = pd.concat([t["ncurves"] for t in tables], ignore_index=True)
matched_groups = pd.concat([t["matched_groups"] for t in tables], ignore_index=True)
RUN_DIRS

## Headline metrics (weighted, thermometer ballots)

In [ ]:
headline = [
    "has_cycle",
    "condorcet_winner_exists",
    "smith_size",
    "sp_deletion_share",
    "vr_mean_min_violation",
    "pc1_share",
    "mean_kendall_tau",
    "rule_disagreement_share",
]
view = metrics[(metrics["weighting"] == "weighted") & metrics["metric"].isin(headline)]
view = view[(view["source"] == "thermometer")]
view.pivot_table(index=["election", "electorate"], columns="metric", values="value")[
    headline
].round(3)

## Figure 1: cycle-rate n-curves by electorate

In [ ]:
fig1 = analysis.fig_cycle_ncurves(ncurves)

## Figure 2: SP deletion share by electorate × election

In [ ]:
fig2 = analysis.fig_sp_deletion(metrics)

## Figure 3: PC1 share, human vs. LLM

In [ ]:
fig3 = analysis.fig_pc1(metrics)

## Matched human–LLM comparisons by party ID

In [ ]:
by_party = matched_groups[matched_groups["group_var"].isin(["all", "party_id_7"])]
by_party.pivot_table(index=["model", "group"], columns="metric", values="weighted_mean").round(3)